In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import ipywidgets as widgets
from ipywidgets import interactive

# ==========================================
# 1. HODGKIN-HUXLEY EQUATIONS
# ==========================================

def alpha_m(V): return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0)) if V != -40.0 else 1.0
def beta_m(V):  return 4.0 * np.exp(-(V + 65.0) / 18.0)
def alpha_h(V): return 0.07 * np.exp(-(V + 65.0) / 20.0)
def beta_h(V):  return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
def alpha_n(V): return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0)) if V != -55.0 else 0.1
def beta_n(V):  return 0.125 * np.exp(-(V + 65.0) / 80.0)

# Injected current function: constant stimulus over a time window
def I_inj(t, I_app):
    return I_app if 10.0 < t < 400.0 else 0.0

# Derivative vector function for odeint
def dALLdt(X, t, g_Na, g_K, g_L, E_Na, E_K, E_L, I_app, C_m):
    V, m, h, n = X

    # Calculate currents
    I_Na = g_Na * (m**3) * h * (V - E_Na)
    I_K  = g_K * (n**4) * (V - E_K)
    I_L  = g_L * (V - E_L)

    # Differential equations
    dVdt = (I_inj(t, I_app) - I_Na - I_K - I_L) / C_m
    dmdt = alpha_m(V) * (1.0 - m) - beta_m(V) * m
    dhdt = alpha_h(V) * (1.0 - h) - beta_h(V) * h
    dndt = alpha_n(V) * (1.0 - n) - beta_n(V) * n

    return dVdt, dmdt, dhdt, dndt

# ==========================================
# 2. INTERACTIVE PLOTTING FUNCTION
# ==========================================

def simulate_hh(g_Na=120.0, g_K=36.0, g_L=0.3, E_Na=50.0, E_K=-77.0, E_L=-54.387, I_app=10.0, C_m=1.0):
    t = np.arange(0.0, 100.0, 0.05) # Simulated time array (0-100 ms)

    # Initial conditions: [V, m, h, n] at rest
    X0 = [-65.0, 0.05, 0.6, 0.32]

    # Integrate equations with chosen slider parameters
    X = odeint(dALLdt, X0, t, args=(g_Na, g_K, g_L, E_Na, E_K, E_L, I_app, C_m))
    V = X[:, 0]
    m = X[:, 1]
    h = X[:, 2]
    n = X[:, 3]

    # Generate the layout plots
    plt.figure(figsize=(12, 8))

    # Top Plot: Membrane Voltage
    plt.subplot(2, 1, 1)
    plt.plot(t, V, 'k', lw=2)
    plt.title('Interactive Hodgkin-Huxley Model Simulation', fontsize=14)
    plt.ylabel('Membrane Potential V (mV)', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.6)

    # Bottom Plot: Gating Variables
    plt.subplot(2, 1, 2)
    plt.plot(t, m, 'g', label='m (Na activation)')
    plt.plot(t, h, 'r', label='h (Na inactivation)')
    plt.plot(t, n, 'b', label='n (K activation)')
    plt.ylabel('Gating Variable Probability', fontsize=11)
    plt.xlabel('Time (ms)', fontsize=11)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# ==========================================
# 3. BUILD INTERACTIVE WIDGET CONTROLS
# ==========================================

interactive_plot = interactive(
    simulate_hh,
    g_Na=widgets.FloatSlider(value=120.0, min=0.0, max=200.0, step=5.0, description='g_Na (mS/cm²)'),
    g_K=widgets.FloatSlider(value=36.0, min=0.0, max=100.0, step=2.0, description='g_K (mS/cm²)'),
    g_L=widgets.FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description='g_L (Leak)'),
    E_Na=widgets.FloatSlider(value=50.0, min=20.0, max=80.0, step=2.0, description='E_Na (mV)'),
    E_K=widgets.FloatSlider(value=-77.0, min=-90.0, max=-50.0, step=2.0, description='E_K (mV)'),
    E_L=widgets.FloatSlider(value=-54.387, min=-70.0, max=-40.0, step=2.0, description='E_L (mV)'),
    I_app=widgets.FloatSlider(value=10.0, min=0.0, max=50.0, step=1.0, description='I_injected'),
    C_m=widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.1, description='Capacitance C_m')
)

# Render sliders and initial graph output layout
display(interactive_plot)


interactive(children=(FloatSlider(value=120.0, description='g_Na (mS/cm²)', max=200.0, step=5.0), FloatSlider(…